In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# T46 current-clean hinge / tanh 对照（用户运行）

固定两原dev内容seed，OFF/HINGE_A/B/TANH_A/B，共10个正式视频。
载体/码本/T46/原native UniPC/旧reader保持。温度1 tanh竞争目标与原hinge，仅当前clean46代理梯度。
复用指定原run SHA验证后的z46/v46、完整history、prompt和book；不重采样prefix、不重编码prompt。
每臂独立单位nativeprobe，匹配原LOCAL实际nativeD预算；绝不用离线terminal残差RMS冒充nativeD。
正式OFF也从原46状态独立零控制续程，随后47–49当前model自由采样，与新arms构成同批baseline。
原模型revision未记录：记录新加载revision与旧OFF恢复差异，但不声称原权重身份相同。
这是saved46条件比较，不是同权重完整重生成，也不是终态梯度transport。

全轮60TF /40native /8unit probes /8CPU clean-leaf backwards；无模型或VAE反传。
全部10新视频decode/save，floatRGB、RGB8、真正MP4三层各4相位，120次新VAE encode。
原终态仅名义统计，不伪造四相位。原OFF只作历史兼容性参考；质量/竞争gain全部对新OFF。
对照四pair固定，缺失/失败保留10/120分母，无扫描、无参数择优、无旧tau成功门槛。
查看paired_summary：terminal和各层correct/wrong绝对score、gap、tanh−hinge及相对新OFF gain。
竞争改善不等于存在性、感知质量或轨迹成功；请人工检查内容/伪影/运动。
实现仅CPU/fake检查，agent未运行真实模型。Run all执行固定实验并保存完整记录。
固定原run：flow_tube_response_selection_20260921T013844126172Z。
源码SHA：c82e931b2fbd66644a3d77a03e7c328b52d1a2e9。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = 'c82e931b2fbd66644a3d77a03e7c328b52d1a2e9'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'flow_tube_clean_proxy_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
INPUT = Path('/content/drive/MyDrive/Video-WM/FlowTubeResponseSelection/flow_tube_response_selection_20260921T013844126172Z')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/FlowTubeCleanProxy') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.flow_tube_clean_proxy_run', '--source', str(INPUT), '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({k:result.get(k) for k in ('status','video_denominator','receiver_encode_denominator','fixed_calls','actual_calls_observed','media_layer_denominator','historical_OFF_compatibility','paired_summary')}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
